# OLIST — Data Connection, Cleaning & Validation

### Objective
Connect to the OLIST PostgreSQL database, load the analytical tables, validate data quality, clean analytical fields, and prepare reusable datasets for business analysis.

### Data Areas
- Customers
- Geolocation
- Orders
- Order Items
- Payments
- Reviews
- Products
- Sellers
- Product Category Translation

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, text
from dotenv import load_dotenv

import os

In [2]:
load_dotenv()

USE_CSV_FALLBACK = True

if not USE_CSV_FALLBACK:
    user = os.getenv("POSTGRES_USER")
    password = os.getenv("POSTGRES_PASSWORD")
    host = os.getenv("POSTGRES_HOST", "localhost")
    port = os.getenv("POSTGRES_PORT", "5432")
    database = os.getenv("POSTGRES_DB", "olist")

    engine = create_engine(
        f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
    )

    with engine.connect() as connection:
        connection.execute(text("SELECT 1"))

    print("PostgreSQL connection successful!")

In [3]:
table_names = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
]

if not USE_CSV_FALLBACK:
    tables = {}

    for table in table_names:
        tables[table] = pd.read_sql(
            f"SELECT * FROM {table};",
            engine
        )

    customers = tables["olist_customers_dataset"]
    geolocation = tables["olist_geolocation_dataset"]
    orders = tables["olist_orders_dataset"]
    order_items = tables["olist_order_items_dataset"]
    payments = tables["olist_order_payments_dataset"]
    reviews = tables["olist_order_reviews_dataset"]
    products = tables["olist_products_dataset"]
    sellers = tables["olist_sellers_dataset"]
    translation = tables["product_category_name_translation"]

In [4]:
if USE_CSV_FALLBACK:
    data_path = "."

    customers = pd.read_csv(
        f"{data_path}/olist_customers_dataset.csv"
    )
    geolocation = pd.read_csv(
        f"{data_path}/olist_geolocation_dataset.csv"
    )
    orders = pd.read_csv(
        f"{data_path}/olist_orders_dataset.csv"
    )
    order_items = pd.read_csv(
        f"{data_path}/olist_order_items_dataset.csv"
    )
    payments = pd.read_csv(
        f"{data_path}/olist_order_payments_dataset.csv"
    )
    reviews = pd.read_csv(
        f"{data_path}/olist_order_reviews_dataset.csv"
    )
    products = pd.read_csv(
        f"{data_path}/olist_products_dataset.csv"
    )
    sellers = pd.read_csv(
        f"{data_path}/olist_sellers_dataset.csv"
    )
    translation = pd.read_csv(
        f"{data_path}/product_category_name_translation.csv"
    )

In [5]:
row_counts = pd.Series({
    "Customers": len(customers),
    "Geolocation": len(geolocation),
    "Orders": len(orders),
    "Order Items": len(order_items),
    "Payments": len(payments),
    "Reviews": len(reviews),
    "Products": len(products),
    "Sellers": len(sellers),
    "Category Translation": len(translation)
})

row_counts

Customers                 99441
Geolocation             1000163
Orders                    99441
Order Items              112650
Payments                 103886
Reviews                   99224
Products                  32951
Sellers                    3095
Category Translation         71
dtype: int64

In [6]:
table_shapes = pd.DataFrame({
    "table": table_names,
    "rows": [
        len(customers),
        len(geolocation),
        len(orders),
        len(order_items),
        len(payments),
        len(reviews),
        len(products),
        len(sellers),
        len(translation)
    ],
    "columns": [
        len(customers.columns),
        len(geolocation.columns),
        len(orders.columns),
        len(order_items.columns),
        len(payments.columns),
        len(reviews.columns),
        len(products.columns),
        len(sellers.columns),
        len(translation.columns)
    ]
})

table_shapes

,table,rows,columns
0,olist_customers_dataset,99441,5
1,olist_geolocation_dataset,1000163,5
2,olist_orders_dataset,99441,8
3,olist_order_items_dataset,112650,7
4,olist_order_payments_dataset,103886,5
5,olist_order_reviews_dataset,99224,7
6,olist_products_dataset,32951,9
7,olist_sellers_dataset,3095,4
8,product_category_name_translation,71,2


In [7]:
orders.dtypes
order_items.dtypes
payments.dtypes
reviews.dtypes
products.dtypes

product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object

In [8]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

order_items["shipping_limit_date"] = pd.to_datetime(
    order_items["shipping_limit_date"],
    errors="coerce"
)

review_date_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in review_date_columns:
    reviews[column] = pd.to_datetime(
        reviews[column],
        errors="coerce"
    )

In [9]:
missing_values = pd.DataFrame({
    "customers": customers.isna().sum(),
    "orders": orders.isna().sum(),
    "order_items": order_items.isna().sum(),
    "payments": payments.isna().sum(),
    "reviews": reviews.isna().sum(),
    "products": products.isna().sum()
})

missing_values

,customers,orders,order_items,payments,reviews,products
customer_city,0.0,NaN,NaN,NaN,NaN,NaN
customer_id,0.0,0.0,NaN,NaN,NaN,NaN
customer_state,0.0,NaN,NaN,NaN,NaN,NaN
customer_unique_id,0.0,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,0.0,NaN,NaN,NaN,NaN,NaN
freight_value,NaN,NaN,0.0,NaN,NaN,NaN
order_approved_at,NaN,160.0,NaN,NaN,NaN,NaN
order_delivered_carrier_date,NaN,1783.0,NaN,NaN,NaN,NaN
order_delivered_customer_date,NaN,2965.0,NaN,NaN,NaN,NaN
order_estimated_delivery_date,NaN,0.0,NaN,NaN,NaN,NaN


In [10]:
missing_by_table = pd.Series({
    "Customers": customers.isna().sum().sum(),
    "Geolocation": geolocation.isna().sum().sum(),
    "Orders": orders.isna().sum().sum(),
    "Order Items": order_items.isna().sum().sum(),
    "Payments": payments.isna().sum().sum(),
    "Reviews": reviews.isna().sum().sum(),
    "Products": products.isna().sum().sum(),
    "Sellers": sellers.isna().sum().sum(),
    "Translation": translation.isna().sum().sum()
})

missing_by_table

Customers           0
Geolocation         0
Orders           4908
Order Items         0
Payments            0
Reviews        145903
Products         2448
Sellers             0
Translation         0
dtype: int64

In [11]:
duplicate_counts = pd.Series({
    "Customers": customers.duplicated().sum(),
    "Geolocation": geolocation.duplicated().sum(),
    "Orders": orders.duplicated().sum(),
    "Order Items": order_items.duplicated().sum(),
    "Payments": payments.duplicated().sum(),
    "Reviews": reviews.duplicated().sum(),
    "Products": products.duplicated().sum(),
    "Sellers": sellers.duplicated().sum(),
    "Translation": translation.duplicated().sum()
})

duplicate_counts

Customers           0
Geolocation    261831
Orders              0
Order Items         0
Payments            0
Reviews             0
Products            0
Sellers             0
Translation         0
dtype: int64

In [12]:
primary_key_checks = pd.DataFrame({
    "table": [
        "customers",
        "products",
        "sellers",
        "orders",
        "translation"
    ],
    "key": [
        "customer_id",
        "product_id",
        "seller_id",
        "order_id",
        "product_category_name"
    ],
    "duplicate_keys": [
        customers["customer_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        sellers["seller_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        translation["product_category_name"].duplicated().sum()
    ]
})

primary_key_checks

,table,key,duplicate_keys
0,customers,customer_id,0
1,products,product_id,0
2,sellers,seller_id,0
3,orders,order_id,0
4,translation,product_category_name,0


In [13]:
invalid_order_customers = (
    orders.loc[
        ~orders["customer_id"].isin(customers["customer_id"]),
        ["order_id", "customer_id"]
    ]
)

invalid_order_customers.head()

,order_id,customer_id


In [14]:
invalid_item_orders = (
    order_items.loc[
        ~order_items["order_id"].isin(orders["order_id"]),
        ["order_id"]
    ]
)

invalid_item_orders.head()

,order_id


In [15]:
invalid_item_products = (
    order_items.loc[
        ~order_items["product_id"].isin(products["product_id"]),
        ["product_id"]
    ]
)

invalid_item_products.head()

,product_id


In [16]:
invalid_item_sellers = (
    order_items.loc[
        ~order_items["seller_id"].isin(sellers["seller_id"]),
        ["seller_id"]
    ]
)

invalid_item_sellers.head()

,seller_id


In [17]:
invalid_payment_orders = (
    payments.loc[
        ~payments["order_id"].isin(orders["order_id"]),
        ["order_id"]
    ]
)

invalid_payment_orders.head()

,order_id


In [18]:
invalid_review_orders = (
    reviews.loc[
        ~reviews["order_id"].isin(orders["order_id"]),
        ["order_id"]
    ]
)

invalid_review_orders.head()

,order_id


In [19]:
orders["order_status"].value_counts(dropna=False)

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [20]:
payments["payment_type"].value_counts(dropna=False)

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

In [21]:
reviews["review_score"].value_counts(dropna=False).sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [22]:
numeric_validation = pd.Series({
    "negative_price": (order_items["price"] < 0).sum(),
    "negative_freight": (order_items["freight_value"] < 0).sum(),
    "invalid_review_score": (
        (~reviews["review_score"].between(1, 5)) &
        reviews["review_score"].notna()
    ).sum(),
    "negative_payment_value": (payments["payment_value"] < 0).sum(),
    "invalid_payment_installments": (
        (payments["payment_installments"] <= 0) &
        payments["payment_installments"].notna()
    ).sum()
})

numeric_validation

negative_price                  0
negative_freight                0
invalid_review_score            0
negative_payment_value          0
invalid_payment_installments    2
dtype: int64

In [23]:
date_validation = pd.Series({
    "approved_before_purchase": (
        orders["order_approved_at"] < orders["order_purchase_timestamp"]
    ).sum(),
    "carrier_before_purchase": (
        orders["order_delivered_carrier_date"] < orders["order_purchase_timestamp"]
    ).sum(),
    "customer_delivery_before_purchase": (
        orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
    ).sum(),
    "estimated_before_purchase": (
        orders["order_estimated_delivery_date"] < orders["order_purchase_timestamp"]
    ).sum()
})

date_validation

approved_before_purchase               0
carrier_before_purchase              166
customer_delivery_before_purchase      0
estimated_before_purchase              0
dtype: int64

In [24]:
text_columns = [
    "customer_city",
    "customer_state"
]

for column in text_columns:
    customers[column] = (
        customers[column]
        .astype("string")
        .str.strip()
    )

for column in ["seller_city", "seller_state"]:
    sellers[column] = (
        sellers[column]
        .astype("string")
        .str.strip()
    )

products["product_category_name"] = (
    products["product_category_name"]
    .astype("string")
    .str.strip()
)

translation["product_category_name"] = (
    translation["product_category_name"]
    .astype("string")
    .str.strip()
)

translation["product_category_name_english"] = (
    translation["product_category_name_english"]
    .astype("string")
    .str.strip()
)

In [25]:
numeric_columns = [
    "price",
    "freight_value"
]

for column in numeric_columns:
    order_items[column] = pd.to_numeric(
        order_items[column],
        errors="coerce"
    )

payments["payment_value"] = pd.to_numeric(
    payments["payment_value"],
    errors="coerce"
)

payments["payment_installments"] = pd.to_numeric(
    payments["payment_installments"],
    errors="coerce"
)

products_numeric = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
]

for column in products_numeric:
    products[column] = pd.to_numeric(
        products[column],
        errors="coerce"
    )

In [26]:
products_analysis = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products_analysis["category"] = (
    products_analysis["product_category_name_english"]
    .fillna(products_analysis["product_category_name"])
)

products_analysis["volume_cm3"] = (
    products_analysis["product_length_cm"] *
    products_analysis["product_height_cm"] *
    products_analysis["product_width_cm"]
)

products_analysis.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,category,volume_cm3
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0,perfumery,perfumery,2240.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0,art,art,10800.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0,sports_leisure,sports_leisure,2430.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0,baby,baby,2704.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0,housewares,housewares,4420.0


In [27]:
reviews_latest = (
    reviews
    .sort_values(
        ["order_id", "review_creation_date"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["order_id"],
        keep="first"
    )
)

reviews_latest = reviews_latest[
    ["order_id", "review_score", "review_creation_date"]
]

reviews_latest.head()

,order_id,review_score,review_creation_date
51963,00010242fe8c5a6d1ba2dd792cb16214,5,2017-09-21
27823,00018f77f2f0320c557190d7a144bdd3,4,2017-05-13
4218,000229ec398224ef6ca0657da4fc703e,5,2018-01-23
38844,00024acbcdf0a6daa1e931b038114c75,4,2018-08-15
55676,00042b26cf59d7ce69dfabb4e55b4fd9,5,2017-03-02


In [28]:
order_metrics = (
    order_items
    .groupby("order_id", as_index=False)
    .agg(
        order_revenue=("price", "sum"),
        freight_value=("freight_value", "sum"),
        item_count=("order_item_id", "count")
    )
)

order_metrics["order_total_before_payment"] = (
    order_metrics["order_revenue"] +
    order_metrics["freight_value"]
)

order_metrics.head()

,order_id,order_revenue,freight_value,item_count,order_total_before_payment
0,00010242fe8c5a6d1ba2dd792cb16214,58.90,13.29,1,72.19
1,00018f77f2f0320c557190d7a144bdd3,239.90,19.93,1,259.83
2,000229ec398224ef6ca0657da4fc703e,199.00,17.87,1,216.87
3,00024acbcdf0a6daa1e931b038114c75,12.99,12.79,1,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,199.90,18.14,1,218.04


In [29]:
order_analysis = (
    orders
    .merge(
        customers[
            ["customer_id", "customer_unique_id",
             "customer_city", "customer_state"]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        order_metrics,
        on="order_id",
        how="left"
    )
    .merge(
        reviews_latest[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="left"
    )
)

order_analysis["order_revenue"] = (
    order_analysis["order_revenue"].fillna(0)
)

order_analysis["freight_value"] = (
    order_analysis["freight_value"].fillna(0)
)

order_analysis["item_count"] = (
    order_analysis["item_count"].fillna(0)
)

order_analysis["delivery_days"] = (
    order_analysis["order_delivered_customer_date"] -
    order_analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

order_analysis["is_late"] = (
    (order_analysis["order_status"] == "delivered") &
    order_analysis["order_delivered_customer_date"].notna() &
    order_analysis["order_estimated_delivery_date"].notna() &
    (
        order_analysis["order_delivered_customer_date"] >
        order_analysis["order_estimated_delivery_date"]
    )
)

order_analysis.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,order_revenue,freight_value,item_count,order_total_before_payment,review_score,delivery_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,sao paulo,SP,29.99,8.72,1.0,38.71,4.0,8.436574,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,barreiras,BA,118.70,22.76,1.0,141.46,4.0,13.782037,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,vianopolis,GO,159.90,19.22,1.0,179.12,5.0,9.394213,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,sao goncalo do amarante,RN,45.00,27.20,1.0,72.20,5.0,13.208750,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,santo andre,SP,19.90,8.72,1.0,28.62,5.0,2.873877,False


In [30]:
payment_analysis = (
    payments
    .groupby("order_id", as_index=False)
    .agg(
        order_payment_value=("payment_value", "sum"),
        order_payment_installments=("payment_installments", "max"),
        payment_methods_used=("payment_type", "nunique")
    )
)

payment_analysis.head()

,order_id,order_payment_value,order_payment_installments,payment_methods_used
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3,1


In [31]:
sales_analysis = (
    order_items
    .merge(
        products_analysis[
            ["product_id", "category"]
        ],
        on="product_id",
        how="left"
    )
    .merge(
        sellers[
            ["seller_id", "seller_city", "seller_state"]
        ],
        on="seller_id",
        how="left"
    )
    .merge(
        orders[
            ["order_id", "customer_id", "order_status",
             "order_purchase_timestamp",
             "order_delivered_customer_date",
             "order_estimated_delivery_date"]
        ],
        on="order_id",
        how="left"
    )
    .merge(
        customers[
            ["customer_id", "customer_unique_id",
             "customer_city", "customer_state"]
        ],
        on="customer_id",
        how="left"
    )
    .merge(
        reviews_latest[
            ["order_id", "review_score"]
        ],
        on="order_id",
        how="left"
    )
)

sales_analysis["delivery_days"] = (
    sales_analysis["order_delivered_customer_date"] -
    sales_analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

sales_analysis["is_late"] = (
    (sales_analysis["order_status"] == "delivered") &
    sales_analysis["order_delivered_customer_date"].notna() &
    sales_analysis["order_estimated_delivery_date"].notna() &
    (
        sales_analysis["order_delivered_customer_date"] >
        sales_analysis["order_estimated_delivery_date"]
    )
)

sales_analysis.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,category,seller_city,seller_state,customer_id,order_status,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_city,customer_state,review_score,delivery_days,is_late
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,volta redonda,SP,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,campos dos goytacazes,RJ,5.0,7.614421,False
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,sao paulo,SP,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,santa fe do sul,SP,4.0,16.216181,False
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,borda da mata,MG,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,para de minas,MG,5.0,7.948437,False
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,franca,SP,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,atibaia,SP,4.0,6.147269,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,loanda,PR,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,varzea paulista,SP,5.0,25.114352,False


In [32]:
order_analysis["purchase_date"] = (
    order_analysis["order_purchase_timestamp"].dt.date
)

order_analysis["purchase_month"] = (
    order_analysis["order_purchase_timestamp"]
    .dt.to_period("M")
    .astype(str)
)

order_analysis["purchase_quarter"] = (
    order_analysis["order_purchase_timestamp"]
    .dt.to_period("Q")
    .astype(str)
)

order_analysis["purchase_day"] = (
    order_analysis["order_purchase_timestamp"].dt.day_name()
)

order_analysis["purchase_hour"] = (
    order_analysis["order_purchase_timestamp"].dt.hour
)

order_analysis["is_weekend"] = (
    order_analysis["order_purchase_timestamp"].dt.dayofweek >= 5
)

In [33]:
order_analysis["order_value_band"] = pd.cut(
    order_analysis["order_revenue"],
    bins=[
        -np.inf,
        50,
        200,
        500,
        1000,
        np.inf
    ],
    labels=[
        "Low",
        "Medium",
        "Upper Medium",
        "High",
        "Very High"
    ]
)

order_analysis["order_value_band"].value_counts(
    sort=False
)

order_value_band
Low             30429
Medium          54020
Upper Medium    11375
High             2674
Very High         943
Name: count, dtype: int64

In [34]:
order_analysis["delivery_speed_band"] = pd.cut(
    order_analysis["delivery_days"],
    bins=[
        -np.inf,
        3,
        7,
        14,
        30,
        np.inf
    ],
    labels=[
        "0-3 Days",
        "4-7 Days",
        "8-14 Days",
        "15-30 Days",
        "31+ Days"
    ]
)

order_analysis["delivery_speed_band"].value_counts(
    sort=False
)

delivery_speed_band
0-3 Days       4753
4-7 Days      21293
8-14 Days     40215
15-30 Days    25662
31+ Days       4553
Name: count, dtype: int64

In [35]:
customer_order_metrics = (
    order_analysis[
        order_analysis["order_status"] == "delivered"
    ]
    .groupby("customer_unique_id", as_index=False)
    .agg(
        order_count=("order_id", "nunique"),
        total_revenue=("order_revenue", "sum"),
        average_order_value=("order_revenue", "mean")
    )
)

customer_order_metrics["customer_type"] = np.where(
    customer_order_metrics["order_count"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

customer_order_metrics.head()

,customer_unique_id,order_count,total_revenue,average_order_value,customer_type
0,0000366f3b9a7992bf8c76cfdf3221e2,1,129.90,129.90,One-Time Customer
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,18.90,18.90,One-Time Customer
2,0000f46a3911fa3c0805444483337064,1,69.00,69.00,One-Time Customer
3,0000f6ccb0745a6a4b88665a16c9f078,1,25.99,25.99,One-Time Customer
4,0004aac84e0df4da2b147fca70cf8255,1,180.00,180.00,One-Time Customer


In [36]:
seller_metrics = (
    sales_analysis[
        sales_analysis["order_status"] == "delivered"
    ]
    .groupby("seller_id", as_index=False)
    .agg(
        total_revenue=("price", "sum"),
        total_items=("order_item_id", "count"),
        average_item_price=("price", "mean"),
        average_review_score=("review_score", "mean"),
        late_item_rate=("is_late", "mean")
    )
)

seller_metrics["late_item_rate"] = (
    seller_metrics["late_item_rate"] * 100
)

seller_metrics.head()

,seller_id,total_revenue,total_items,average_item_price,average_review_score,late_item_rate
0,0015a82c2db000af6aaaf3ae2ecb0532,2685.00,3,895.000000,3.666667,0.000000
1,001cca7ae9ae17fb1caed9dfb1094831,24487.03,234,104.645427,3.965368,5.555556
2,002100f778ceb8431b7a1020ff7ab48f,1216.60,54,22.529630,4.037037,16.666667
3,003554e2dce176b5555353e4f3555ac8,120.00,1,120.000000,5.000000,0.000000
4,004c9cd9d87a3c30c522c48c4fc07416,19569.73,168,116.486488,4.150602,7.738095


In [37]:
category_metrics = (
    sales_analysis[
        sales_analysis["order_status"] == "delivered"
    ]
    .groupby("category", as_index=False)
    .agg(
        revenue=("price", "sum"),
        items=("order_item_id", "count"),
        average_price=("price", "mean"),
        average_review_score=("review_score", "mean"),
        freight_value=("freight_value", "sum")
    )
    .sort_values("revenue", ascending=False)
)

category_metrics["freight_to_revenue_pct"] = (
    category_metrics["freight_value"] /
    category_metrics["revenue"] *
    100
)

category_metrics.head()

,category,revenue,items,average_price,average_review_score,freight_value,freight_to_revenue_pct
43,health_beauty,1233131.72,9465,130.283330,4.189853,178957.81,14.512465
72,watches_gifts,1166176.98,5859,199.040276,4.071711,98156.14,8.416916
7,bed_bath_table,1023434.76,10953,93.438762,3.923183,201774.50,19.715424
67,sports_leisure,954852.55,8431,113.254958,4.165433,163404.36,17.113046
15,computers_accessories,888724.61,7644,116.264339,3.986725,143999.16,16.202900


In [38]:
validation_summary = pd.DataFrame({
    "check": [
        "Duplicate customers",
        "Duplicate products",
        "Duplicate sellers",
        "Duplicate orders",
        "Orphan order customers",
        "Orphan order-item orders",
        "Orphan order-item products",
        "Orphan order-item sellers",
        "Orphan payment orders",
        "Orphan review orders",
        "Negative item prices",
        "Negative freight values",
        "Invalid review scores"
    ],
    "count": [
        customers["customer_id"].duplicated().sum(),
        products["product_id"].duplicated().sum(),
        sellers["seller_id"].duplicated().sum(),
        orders["order_id"].duplicated().sum(),
        len(invalid_order_customers),
        len(invalid_item_orders),
        len(invalid_item_products),
        len(invalid_item_sellers),
        len(invalid_payment_orders),
        len(invalid_review_orders),
        (order_items["price"] < 0).sum(),
        (order_items["freight_value"] < 0).sum(),
        (
            (~reviews["review_score"].between(1, 5)) &
            reviews["review_score"].notna()
        ).sum()
    ]
})

validation_summary

,check,count
0,Duplicate customers,0
1,Duplicate products,0
2,Duplicate sellers,0
3,Duplicate orders,0
4,Orphan order customers,0
5,Orphan order-item orders,0
6,Orphan order-item products,0
7,Orphan order-item sellers,0
8,Orphan payment orders,0
9,Orphan review orders,0
